# GramSathi - Bias & Fairness Analysis

Analyzing model fairness across demographic groups to ensure equitable outcomes.

**Metrics:**
- Demographic parity difference
- Equal opportunity difference
- Disparate impact ratio
- Bias mitigation techniques

**Protected Attributes:** gender, caste_category, age_group, is_farmer

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.preprocessing import StandardScaler

sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

print('Libraries loaded')

In [ ]:
import sys
sys.path.append('..')
from trainer.data_pipeline import (
    generate_training_dataset, generate_synthetic_citizen_data,
    compute_eligibility_labels, build_preprocessing_pipeline,
    load_from_dataframe, TARGET_COLUMNS, FEATURES
)

df = generate_training_dataset(n_samples=100000, seed=42)
X_train, X_test, y_train, y_test = load_from_dataframe(df)
X_test_orig = X_test.copy()

preprocessor = build_preprocessing_pipeline()
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

model = MultiOutputClassifier(GradientBoostingClassifier(n_estimators=200, max_depth=5, random_state=42), n_jobs=-1)
model.fit(X_train_processed, y_train)
y_pred = model.predict(X_test_processed)

print('Model trained and predictions generated.')
print(f'Test set size: {len(X_test)}')

## 1. Demographic Parity Analysis

**Demographic parity**: The probability of a positive prediction should be equal across groups.

Difference = P(Ŷ=1 | group A) - P(Ŷ=1 | group B)

In [ ]:
protected_attributes = ['gender', 'caste_category', 'is_farmer']

demographic_parity = {}

for attr in protected_attributes:
    if attr not in X_test_orig.columns:
        continue
    
    groups = X_test_orig[attr].unique()
    print(f'\n=== Demographic Parity: {attr} ===')
    
    for scheme_idx, scheme in enumerate(TARGET_COLUMNS):
        pos_rates = {}
        for group in groups:
            mask = X_test_orig[attr] == group
            if mask.sum() == 0:
                continue
            pos_rate = y_pred[mask, scheme_idx].mean()
            pos_rates[str(group)] = float(pos_rate)
        
        demographic_parity[f'{attr}_{scheme}'] = pos_rates
        
        rates = list(pos_rates.values())
        if len(rates) >= 2:
            max_diff = max(rates) - min(rates)
            print(f'  {scheme[:18]:18s} max diff: {max_diff:.4f}', end='')
            if max_diff > 0.1:
                print(' *** POTENTIAL BIAS ***')
            else:
                print()

In [ ]:
fig, axes = plt.subplots(1, len(protected_attributes), figsize=(18, 5))

for attr_idx, attr in enumerate(protected_attributes):
    ax = axes[attr_idx]
    
    scheme_rates = {}
    for scheme in TARGET_COLUMNS:
        key = f'{attr}_{scheme}'
        if key in demographic_parity:
            scheme_rates[scheme.replace('_', ' ').title()[:15]] = demographic_parity[key]
    
    df_plot = pd.DataFrame(scheme_rates).T
    df_plot.plot(kind='bar', ax=ax, colormap='Set2', width=0.8)
    ax.set_title(f'Demographic Parity by {attr}', fontsize=12)
    ax.set_ylabel('Positive Prediction Rate')
    ax.legend(loc='best', fontsize=8)
    ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 2. Equal Opportunity Analysis

**Equal opportunity**: True positive rates should be equal across groups.

Difference = TPR(group A) - TPR(group B)

In [ ]:
equal_opportunity = {}

for attr in protected_attributes:
    if attr not in X_test_orig.columns:
        continue
    
    groups = X_test_orig[attr].unique()
    print(f'\n=== Equal Opportunity: {attr} ===')
    
    for scheme_idx, scheme in enumerate(TARGET_COLUMNS):
        tpr_values = {}
        for group in groups:
            mask = X_test_orig[attr] == group
            y_true_g = y_test.iloc[:, scheme_idx][mask]
            y_pred_g = y_pred[mask, scheme_idx]
            if y_true_g.sum() > 0:
                tp = ((y_true_g == 1) & (y_pred_g == 1)).sum()
                fn = ((y_true_g == 1) & (y_pred_g == 0)).sum()
                tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
                tpr_values[str(group)] = float(tpr)
            else:
                tpr_values[str(group)] = 0.0
        
        equal_opportunity[f'{attr}_{scheme}'] = tpr_values
        
        tprs = [v for v in tpr_values.values() if v > 0]
        if len(tprs) >= 2:
            max_diff = max(tprs) - min(tprs)
            print(f'  {scheme[:18]:18s} max TPR diff: {max_diff:.4f}', end='')
            if max_diff > 0.1:
                print(' *** POTENTIAL BIAS ***')
            else:
                print()

## 3. Disparate Impact Assessment

**Disparate impact**: Ratio of positive prediction rates between groups.
A ratio below 0.8 or above 1.25 indicates potential disparate impact (80% rule).

In [ ]:
def compute_disparate_impact(pos_rates_dict):
    rates = list(pos_rates_dict.values())
    if len(rates) < 2 or max(rates) == 0:
        return {}
    
    max_rate = max(rates)
    min_rate = min(rates)
    
    groups = list(pos_rates_dict.keys())
    impacts = {}
    for g in groups:
        if max_rate > 0:
            impacts[g] = pos_rates_dict[g] / max_rate
    return impacts

print('=== Disparate Impact Assessment (80% Rule) ===')
print('Ratio < 0.8 means group has <80% of the highest group\'s positive rate\n')

for attr in protected_attributes:
    if attr not in X_test_orig.columns:
        continue
    print(f'\n--- {attr} ---')
    
    for scheme in TARGET_COLUMNS:
        key = f'{attr}_{scheme}'
        if key in demographic_parity:
            impacts = compute_disparate_impact(demographic_parity[key])
            flag = any(v < 0.8 for v in impacts.values())
            prefix = '!!' if flag else '  '
            print(f"{prefix} {scheme[:20]:20s} impacts: {impacts}")

## 4. Performance Metrics by Group

In [ ]:
def compute_group_metrics(X, y_true, y_pred, attr, scheme_idx):
    groups = X[attr].unique()
    metrics = {}
    for group in groups:
        mask = X[attr] == group
        if mask.sum() < 5:
            continue
        yt, yp = y_true.iloc[:, scheme_idx][mask], y_pred[mask, scheme_idx]
        metrics[str(group)] = {
            'n': int(mask.sum()),
            'accuracy': float(accuracy_score(yt, yp)),
            'precision': float(precision_score(yt, yp, zero_division=0)),
            'recall': float(recall_score(yt, yp, zero_division=0)),
            'f1': float(f1_score(yt, yp, zero_division=0)),
            'support': int(yt.sum())
        }
    return metrics

all_metrics = {}
for attr in ['caste_category', 'gender']:
    all_metrics[attr] = {}
    for i, scheme in enumerate(TARGET_COLUMNS):
        all_metrics[attr][scheme] = compute_group_metrics(X_test_orig, y_test, y_pred, attr, i)

df_metrics = []
for attr in all_metrics:
    for scheme in all_metrics[attr]:
        for group, m in all_metrics[attr][scheme].items():
            m['attribute'] = attr
            m['scheme'] = scheme
            m['group'] = group
            df_metrics.append(m)

metrics_df = pd.DataFrame(df_metrics)
metrics_df.head(10)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(22, 10))
axes = axes.flatten()

for i, scheme in enumerate(TARGET_COLUMNS):
    ax = axes[i]
    scheme_data = metrics_df[metrics_df['scheme'] == scheme]
    pivot = scheme_data.pivot(index='group', columns='attribute', values='f1')
    pivot.plot(kind='bar', ax=ax, colormap='Set2', width=0.7)
    ax.set_title(f"F1 by Group - {scheme.replace('_', ' ').title()}", fontsize=10)
    ax.set_ylabel('F1 Score')
    ax.legend(loc='lower right', fontsize=8)
    ax.set_ylim(0, 1)
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Model Performance Across Demographic Groups', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Bias Mitigation Techniques

If bias is detected, apply one or more of the following techniques:

### 5.1 Reweighting (Pre-processing)
Assign higher weights to underrepresented groups during training.

In [ ]:
def compute_sample_weights(X, sensitive_attr, target_col, epsilon=0.01):
    from sklearn.utils.class_weight import compute_class_weight
    
    weights = np.ones(len(X))
    
    groups = X[sensitive_attr].unique()
    for group in groups:
        mask = X[sensitive_attr] == group
        n_group = mask.sum()
        n_total = len(X)
        
        target_in_group = target_col[mask].sum()
        target_total = target_col.sum()
        
        expected_ratio = n_group / n_total
        actual_ratio = target_in_group / target_total if target_total > 0 else expected_ratio
        
        weights[mask] = max(epsilon, expected_ratio / max(epsilon, actual_ratio))
    
    return weights

sample_weights = compute_sample_weights(
    X_train, 'caste_category',
    y_train['education_aid']
)

weighted_model = MultiOutputClassifier(
    GradientBoostingClassifier(n_estimators=200, max_depth=5, random_state=42),
    n_jobs=-1
)

print('Reweighted training would use sample_weight parameter:')
print(f'  Weight range: {sample_weights.min():.2f} to {sample_weights.max():.2f}')
print(f'  Weight applied during model.fit(sample_weight=sample_weights)')

### 5.2 Threshold Adjustment (Post-processing)
Adjust decision thresholds per group to equalize outcomes.

In [ ]:
def find_equal_opportunity_thresholds(y_true, y_prob, sensitive_attr, group, target_tpr):
    from sklearn.metrics import roc_curve
    mask = sensitive_attr == group
    if mask.sum() == 0:
        return 0.5
    fpr, tpr, thresholds = roc_curve(y_true[mask], y_prob[mask])
    idx = np.argmin(np.abs(tpr - target_tpr))
    return thresholds[idx]

print('Threshold adjustment example (education_aid):')
print('  Adjust prediction thresholds per caste category to equalize TPR')
print('  Apply via: pred = proba > group_threshold[group]')

### 5.3 Adversarial Debiasing
Train a model that maximizes accuracy while minimizing the adversary's ability to predict the sensitive attribute.

In [ ]:
try:
    from aif360.algorithms.preprocessing import Reweighing
    from aif360.datasets import BinaryLabelDataset
    AIF360_AVAILABLE = True
except ImportError:
    AIF360_AVAILABLE = False
    print('AIF360 not installed. Install with: pip install aif360')
    print('\nIf installed, adversarial debiasing workflow:')
    print('  1. Create BinaryLabelDataset with protected attribute')
    print('  2. Apply Reweighing to compute instance weights')
    print('  3. Train classifier with computed weights')
    print('  4. Evaluate fairness metrics post-mitigation')

## 6. Fairness Metrics Summary

In [ ]:
print('=' * 70)
print('FAIRNESS ANALYSIS SUMMARY')
print('=' * 70)

bias_detected = False

for attr in protected_attributes:
    if attr not in X_test_orig.columns:
        continue
    
    print(f'\nAttribute: {attr}')
    groups = X_test_orig[attr].unique()
    
    for scheme in TARGET_COLUMNS:
        key_dp = f'{attr}_{scheme}'
        key_eo = f'{attr}_{scheme}'
        
        if key_dp in demographic_parity:
            dp_rates = list(demographic_parity[key_dp].values())
            dp_diff = max(dp_rates) - min(dp_rates)
        else:
            dp_diff = 0
        
        if key_eo in equal_opportunity:
            eo_rates = [v for v in equal_opportunity[key_eo].values() if v > 0]
            eo_diff = max(eo_rates) - min(eo_rates) if len(eo_rates) >= 2 else 0
        else:
            eo_diff = 0
        
        disparate = dp_diff > 0.1 or eo_diff > 0.1
        if disparate:
            bias_detected = True
        
        status = '!! BIAS' if disparate else 'OK'
        print(f'  {scheme:25s} DP diff: {dp_diff:.4f} | EO diff: {eo_diff:.4f} | {status}')

print(f'\n{"Bias detected" if bias_detected else "No significant bias detected"}')
print(f'{"Recommendation: Apply mitigation techniques" if bias_detected else "Model appears fair across groups"}')

In [ ]:
results_output = {
    'demographic_parity': {k: {gk: float(gv) for gk, gv in v.items()} for k, v in demographic_parity.items()},
    'equal_opportunity': {k: {gk: float(gv) for gk, gv in v.items()} for k, v in equal_opportunity.items()},
    'bias_detected': bool(bias_detected),
    'mitigation_suggestions': [
        'Reweighting (pre-processing) - adjust sample weights by group',
        'Threshold adjustment (post-processing) - per-group decision thresholds',
        'Adversarial debiasing - if using neural networks',
        'Collect more diverse training data for underrepresented groups'
    ] if bias_detected else ['No mitigation needed']
}

os.makedirs(MODELS_DIR, exist_ok=True)
MODELS_DIR = "C:\\Users\\User\\gramsathi-ai\\ml\\eligibility-model\\models"
with open(os.path.join(MODELS_DIR, 'fairness_report.json'), 'w') as f:
    json.dump(results_output, f, indent=2)

print('Fairness report saved.')